# Final Project — Growth Status Classification

This notebook is the main entry point for a focused machine-learning project.

The project answers one concrete supervised-learning question:

> **Can simple growth-related measurements classify whether a dog growth record is `normal_growth` or `needs_attention`?**

This notebook is the main file to read for evaluation.

## 1. Data source and target

The experiment uses the committed processed sample:

```text
data/processed/dog_growth_classification_sample.csv
```

Original public data source:

```text
University of Liverpool DataCat
"Growth standard charts for monitoring bodyweight in dogs of different sizes - SUPPORTING DATA"
DOI: https://doi.org/10.17638/datacat.liverpool.ac.uk/377
```

The target is binary:

```text
0 = normal_growth
1 = needs_attention
```

Important limitation: this is **not** veterinary diagnosis. It is a course-aligned educational classification signal derived from the processed public dog-growth sample.

## 2. Leakage control

The processed target was derived from body-condition information. To keep the experiment stricter, the model does **not** use these direct label-source fields as input features:

```text
bcs_recorded
bcs_predicted
bcs_source
growth_status
```

The model uses only simple growth and visit-context features such as age, weight, expected adult weight, sex, preventive-care visit flag, and healthy-pet diagnosis flag.

## Mathematical formulation

This project is a supervised binary classification task.

Each row in the processed dataset represents one dog growth record. The record is converted into a feature vector:

$$x_i = [x_{i1}, x_{i2}, \ldots, x_{ip}]$$

The target variable is binary:

$$y_i \in \{0, 1\}$$

where:

- $0$ means `normal_growth`;
- $1$ means `needs_attention`.

The classifier learns a function:

$$f(x_i) \rightarrow \hat{y}_i$$

For probabilistic models, the useful output is the estimated probability:

$$\hat{p}_i = P(y_i = 1 \mid x_i)$$

A simple decision rule is used to convert probability into a class prediction:

$$\hat{y}_i = 1 \text{ if } \hat{p}_i \ge 0.5, \text{ otherwise } 0$$

The baseline model predicts the majority class. Logistic Regression gives a linear reference model, while Random Forest learns non-linear splits between features such as age, current weight, expected adult weight and the engineered weight ratio.

The main evaluation metrics are:

$$Precision = \frac{TP}{TP + FP}$$

$$Recall = \frac{TP}{TP + FN}$$

$$F1 = 2 \cdot \frac{Precision \cdot Recall}{Precision + Recall}$$

These metrics are used together because accuracy alone can hide important classification mistakes. In this project, false negatives mean `needs_attention` records predicted as `normal_growth`, while false positives mean `normal_growth` records predicted as `needs_attention`.


In [1]:
from pathlib import Path
import subprocess
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src" / "run_growth_status_experiment.py").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

## 3. Reproduce the experiment

The script below trains a baseline model and two supervised classifiers, writes metrics, saves the best model, and produces concrete correct/error/borderline prediction examples.

In [2]:
result = subprocess.run(
    [sys.executable, "src/run_growth_status_experiment.py"],
    cwd=PROJECT_ROOT,
    text=True,
    capture_output=True,
    check=True,
)
print(result.stdout)

GROWTH_STATUS_EXPERIMENT_READY
BEST_EXPERIMENT random_forest
METRICS reports/growth_status/metrics.csv
SUMMARY reports/growth_status/experiment_summary.md
MODEL models/growth_status/growth_status_pipeline.joblib



## 4. Model comparison

The baseline is included to show whether the trained models learn something beyond always predicting the majority class.

In [3]:
report_dir = PROJECT_ROOT / "reports" / "growth_status"

metrics = pd.read_csv(report_dir / "metrics.csv")
metrics

,experiment,accuracy,precision,recall,f1,roc_auc,train_rows,test_rows
0,random_forest,0.8308,0.786159,0.9088,0.843043,0.912492,7500,2500
1,logistic_regression,0.8112,0.770891,0.8856,0.824274,0.891552,7500,2500
2,baseline_majority_class,0.5000,0.000000,0.0000,0.000000,NaN,7500,2500


## 5. Confusion matrix

Rows are actual labels and columns are predicted labels.

In [4]:
confusion = pd.read_csv(report_dir / "confusion_matrix.csv", index_col=0)
confusion

,predicted_normal_growth,predicted_needs_attention
actual_normal_growth,941,309
actual_needs_attention,114,1136


## 6. What the model learned

The feature-importance table helps explain which signals the selected model used most strongly. In this project, the most important signals are expected to be related to weight, age, the weight-to-expected-adult-weight ratio, and visit context.

This does not prove medical correctness. It only shows what the model found useful inside this processed dataset.

In [5]:
feature_importance = pd.read_csv(report_dir / "feature_importance.csv")
feature_importance.head(15)

,feature,value,importance_type
0,numeric__average_adult_breed_weight_kg,0.280615,random_forest_importance
1,numeric__weight_kg,0.247515,random_forest_importance
2,numeric__weight_age_interaction,0.237368,random_forest_importance
3,numeric__weight_to_expected_adult_ratio,0.105193,random_forest_importance
4,numeric__visit_age_months,0.062110,random_forest_importance
5,categorical__healthy_pet_diagnosis_Y,0.024552,random_forest_importance
6,categorical__healthy_pet_diagnosis_N,0.018728,random_forest_importance
7,categorical__gender_MN,0.006798,random_forest_importance
8,categorical__gender_M,0.006231,random_forest_importance
9,categorical__gender_FS,0.004520,random_forest_importance


## 7. Concrete correct predictions

These records show examples where the selected model predicted the correct class.

In [6]:
correct_examples = pd.read_csv(report_dir / "correct_examples.csv")
correct_examples.head(8)

,visit_age_months,weight_kg,average_adult_breed_weight_kg,weight_to_expected_adult_ratio,gender,preventive_care_visit,healthy_pet_diagnosis,actual_label,predicted_label,probability_needs_attention,explanation
0,34.57,12.338,12.569,0.981621,FS,Y,N,normal_growth,normal_growth,0.001571,"Age=34.57 months, weight ratio=0.982 (near exp..."
1,35.52,9.072,12.569,0.721776,FS,Y,N,normal_growth,normal_growth,0.001801,"Age=35.52 months, weight ratio=0.722 (developi..."
2,30.04,8.709,12.569,0.692895,FS,Y,N,normal_growth,normal_growth,0.001821,"Age=30.04 months, weight ratio=0.693 (developi..."
3,30.29,9.798,12.569,0.779537,FS,Y,N,normal_growth,normal_growth,0.001833,"Age=30.29 months, weight ratio=0.780 (developi..."
4,30.26,10.705,12.569,0.851699,FS,Y,N,normal_growth,normal_growth,0.001833,"Age=30.26 months, weight ratio=0.852 (near exp..."
5,30.38,11.612,12.569,0.923860,FS,Y,N,normal_growth,normal_growth,0.001833,"Age=30.38 months, weight ratio=0.924 (near exp..."
6,34.90,9.480,12.569,0.754237,FS,Y,N,normal_growth,normal_growth,0.001833,"Age=34.90 months, weight ratio=0.754 (developi..."
7,31.18,10.342,12.569,0.822818,FS,Y,N,normal_growth,normal_growth,0.001833,"Age=31.18 months, weight ratio=0.823 (developi..."


## 8. Concrete errors

These records are the most important part of the analysis. They show where the model is wrong and help explain the model's limits.

In [7]:
error_examples = pd.read_csv(report_dir / "error_examples.csv")
error_examples.head(8)

,visit_age_months,weight_kg,average_adult_breed_weight_kg,weight_to_expected_adult_ratio,gender,preventive_care_visit,healthy_pet_diagnosis,actual_label,predicted_label,probability_needs_attention,explanation
0,35.40,51.256,35.91,1.427346,MN,N,N,normal_growth,needs_attention,0.938726,"Age=35.40 months, weight ratio=1.427 (high wei..."
1,31.98,45.495,35.91,1.266917,M,N,N,normal_growth,needs_attention,0.930086,"Age=31.98 months, weight ratio=1.267 (high wei..."
2,35.36,45.813,35.91,1.275773,MN,N,N,normal_growth,needs_attention,0.929462,"Age=35.36 months, weight ratio=1.276 (high wei..."
3,30.85,45.949,35.91,1.279560,MN,N,N,normal_growth,needs_attention,0.924909,"Age=30.85 months, weight ratio=1.280 (high wei..."
4,31.19,45.359,35.91,1.263130,MN,N,N,normal_growth,needs_attention,0.922889,"Age=31.19 months, weight ratio=1.263 (high wei..."
5,34.96,40.551,35.91,1.129240,FS,N,N,normal_growth,needs_attention,0.912444,"Age=34.96 months, weight ratio=1.129 (near exp..."
6,26.95,42.229,35.91,1.175968,FS,N,N,normal_growth,needs_attention,0.906110,"Age=26.95 months, weight ratio=1.176 (high wei..."
7,29.80,41.005,35.91,1.141882,M,N,N,normal_growth,needs_attention,0.904634,"Age=29.80 months, weight ratio=1.142 (near exp..."


## 9. Borderline / uncertain examples

These examples are close to the decision boundary. They are useful because they show that some records are difficult to separate with simple tabular features only.

In [8]:
borderline_examples = pd.read_csv(report_dir / "borderline_examples.csv")
borderline_examples.head(8)

,visit_age_months,weight_kg,average_adult_breed_weight_kg,weight_to_expected_adult_ratio,gender,preventive_care_visit,healthy_pet_diagnosis,actual_label,predicted_label,probability_needs_attention,explanation
0,12.01,24.494,35.91,0.682094,MN,Y,N,needs_attention,needs_attention,0.500760,"Age=12.01 months, weight ratio=0.682 (developi..."
1,18.19,29.937,35.91,0.833668,FS,N,Y,normal_growth,normal_growth,0.498679,"Age=18.19 months, weight ratio=0.834 (developi..."
2,33.78,33.566,35.91,0.934726,FS,N,Y,needs_attention,normal_growth,0.497451,"Age=33.78 months, weight ratio=0.935 (near exp..."
3,15.72,17.055,35.91,0.474937,MN,Y,N,normal_growth,needs_attention,0.502715,"Age=15.72 months, weight ratio=0.475 (developi..."
4,17.84,29.665,35.91,0.826093,MN,Y,Y,normal_growth,needs_attention,0.503012,"Age=17.84 months, weight ratio=0.826 (developi..."
5,17.39,29.665,35.91,0.826093,MN,Y,Y,normal_growth,needs_attention,0.503228,"Age=17.39 months, weight ratio=0.826 (developi..."
6,10.04,34.428,35.91,0.958730,FS,N,N,needs_attention,needs_attention,0.503618,"Age=10.04 months, weight ratio=0.959 (near exp..."
7,16.32,30.209,35.91,0.841242,MN,Y,Y,needs_attention,normal_growth,0.495983,"Age=16.32 months, weight ratio=0.841 (developi..."


## 10. Interpretation

The model learns a useful relationship between age, weight, expected adult weight and visit context. It performs much better than the majority-class baseline, which means the features contain signal.

The errors show the main limitation: some records near the boundary between `normal_growth` and `needs_attention` look similar when only simple tabular growth measurements are used. This is why the model should be interpreted as an educational monitoring signal, not as a medical decision system.

For full reproducibility, run:

```bash
python src/run_growth_status_experiment.py
python tests/smoke_test_growth_status_experiment.py
```